In [12]:
import numpy as np
import pandas as pd


In [13]:
attribute_names = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
df = pd.DataFrame(data=load_iris().data, columns=attribute_names[:4])
df['species'] = load_iris().target
class_mapping = {label: idx for idx, label in enumerate(np.unique(df['species']))}
df['species'] = df['species'].map(class_mapping)

X = df.iloc[:, :4].values
y = df['species'].values


In [14]:
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state:
        np.random.seed(random_state)
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)

    test_size = int(X.shape[0] * test_size)
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]

    return X_train, X_test, y_train, y_test


In [15]:
def standardize(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)

    X_train_std = (X_train - mean) / std
    X_test_std = (X_test - mean) / std

    return X_train_std, X_test_std

In [16]:
# Naive Bayes classifier from scratch
class NaiveBayes:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = np.zeros((len(self.classes), X.shape[1]), dtype=np.float64)
        self.variance = np.zeros((len(self.classes), X.shape[1]), dtype=np.float64)
        self.priors = np.zeros(len(self.classes), dtype=np.float64)

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.mean[idx, :] = X_c.mean(axis=0)
            self.variance[idx, :] = X_c.var(axis=0)
            self.priors[idx] = X_c.shape[0] / float(X.shape[0])

    def predict(self, X):
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        posteriors = []

        for idx, c in enumerate(self.classes):
            prior = np.log(self.priors[idx])
            posterior = np.sum(np.log(self._pdf(idx, x)))
            posterior = prior + posterior
            posteriors.append(posterior)

        return self.classes[np.argmax(posteriors)]

    def _pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.variance[class_idx]
        numerator = np.exp(- (x - mean)**2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=82)

X_train_std, X_test_std = standardize(X_train, X_test)

nb = NaiveBayes()
nb.fit(X_train_std, y_train)
y_pred = nb.predict(X_test_std)
print("Predicted labels:", y_pred)

def accuracy_score(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")


Predicted labels: [2 2 0 0 0 2 1 1 1 1 1 2 0 0 0 0 2 1 0 1 0 2 0 2 2 1 2 0 2 1]
Accuracy: 0.93


In [18]:
def confusion_matrix(y_true, y_pred):
    K = len(np.unique(y_true))
    result = np.zeros((K, K))

    for i in range(len(y_true)):
        result[y_true[i]][y_pred[i]] += 1

    return result
y_compare = np.vstack((y_test, y_pred)).T
print("Comparison of test and predicted labels (first 5 samples):\n", y_compare[:5, :])


Comparison of test and predicted labels (first 5 samples):
 [[2 2]
 [2 2]
 [0 0]
 [0 0]
 [0 0]]


In [20]:
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

corrPred = np.trace(cm)
falsePred = np.sum(cm) - corrPred
print('Correct predictions:', corrPred)
print('False predictions:', falsePred)




Confusion Matrix:
 [[11.  0.  0.]
 [ 0.  8.  1.]
 [ 0.  1.  9.]]
Correct predictions: 28.0
False predictions: 2.0


In [21]:
accuracy_from_cm = corrPred / cm.sum()
print('\n\nAccuracy of the Naive Bayes Classification is:', accuracy_from_cm)




Accuracy of the Naive Bayes Classification is: 0.9333333333333333
